In [ ]:
import pandas as pd
import os

import sys
sys.path.append("..")
from src.Eval import aggregated_results_llm, results_to_dataframe, results_for_metric


In [ ]:
%%capture

output_llm = "../output/llm"

results_rdyt = aggregated_results_llm(os.path.join(output_llm, "reddit+shsyt"))
results_yt = aggregated_results_llm(os.path.join(output_llm, "shsyt"))


# Overall Benchmark

In [ ]:
import pandas as pd
import numpy as np

results_for_metric(results_rdyt, "f1")


In [ ]:
results_for_metric(results_yt, "f1")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot(data, sampling_method, metric, baseline_value, baseline_name, 
         eval_cls="overall", eval_scheme="strict", save_path=None):
    # Rename models for cleaner plots
    data = data.rename(index={'Mixtral-8x22B:8x22b': 'Mixtral-8x22B', 
                               'firefunction-v2': 'FireFunction-v2', 
                               'llama3.1:70b': 'Llama3.1-70B', 
                               'gpt-4o-mini': 'GPT-4o-mini'})
    # Filter by sampling method
    data = data[('Value', 'strict', 'overall', 'f1_macro')].to_frame()
    data.columns = ["Value"]
    data = data.reset_index()

    sns.set_style("whitegrid")
    ax = sns.lineplot(data=data, x="k", y="Value", hue='Model', marker='o')

    # Plot the horizontal baseline line
    baseline_line = ax.axhline(y=baseline_value, color='red', linestyle='--', label=baseline_name)

    # Customize the legend
    handles, labels = ax.get_legend_handles_labels()

    # Add the baseline handle and label if not already present
    if baseline_name not in labels:
        baseline_patch = plt.Line2D([0], [0], color='red', linestyle='--', label=baseline_name)
        handles.append(baseline_patch)
        labels.append(baseline_name)

    # Recreate the legend with the updated handles and labels
    ax.legend(handles=handles, labels=labels, title="Model")

    # Custom y-axis label
    plt.ylabel(' '.join(w.capitalize() for w in metric.split("_")))

    if save_path:
        plt.savefig(save_path)

    # Show the plot
    plt.show()


sampling_method = "rand"
metric = "f1_macro"
eval_scheme = "strict"
eval_cls = "overall"
baseline_value = 0.76

plot(results_for_metric(results_rdyt, "f1"), sampling_method, metric, baseline_value, "BERT", save_path="../figures/results_rand_f1_macro.pdf")

